# YawDD Yawning Detection

This experiment explores a computer vision classification task using the YawDD dataset.

The goal is to build a pipeline for detecting yawning behavior from extracted video frames and evaluate the performance of deep learning models on this task.

## Environment Setup

The required dependencies are listed in `requirements.txt`.

## Reset Generated Frames

The frame directory is cleared before each run to avoid mixing outputs from previous preprocessing steps.

In [ ]:
import os
import shutil
from pathlib import Path


frames_dir = Path(r"C:\Users\laith\Desktop\UniAssignments\CV\YawDD\data\output\frames")

for item in frames_dir.iterdir():
    if item.is_dir():
        shutil.rmtree(item)
    else:
        item.unlink()

print(f"Cleared: {frames_dir}")

Cleared: C:\Users\laith\Desktop\UniAssignments\CV\YawDDFinal\data\output\frames


In [2]:
import os
import math

import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

## Project Paths

The experiment uses a structured directory layout for storing raw data, extracted frames, processed crops, and generated metadata files.

In [3]:
from pathlib import Path


# Dataset directory (read-only)
DATASET_ROOT = Path("data/raw")

# Experiment outputs
OUTPUT_ROOT = Path("data/output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


# Generated files
VIDEOS_INFO_PATH = OUTPUT_ROOT / "videos_info.csv"
FRAMES_ROOT = OUTPUT_ROOT / "frames"
FRAMES_INFO_PATH = OUTPUT_ROOT / "frames_info.csv"

CROPS_ROOT = OUTPUT_ROOT / "crops"
CROPS_INFO_PATH = OUTPUT_ROOT / "crops_labeled.csv"

SPLIT_DIR = OUTPUT_ROOT / "splits"


for directory in [FRAMES_ROOT, CROPS_ROOT, SPLIT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def safe_imread(path):
    """
    Load an image and raise an error if the file cannot be read.
    """
    image = cv2.imread(str(path))

    if image is None:
        raise FileNotFoundError(f"Unable to read image: {path}")

    return image

## Dataset Indexing

The raw dataset is scanned to create a metadata file containing video paths, labels, and subject identifiers.

The subject identifier is used later during dataset splitting to prevent frames extracted from the same video from appearing in both training and validation sets.

In [4]:
import pandas as pd
from pathlib import Path


video_rows = []

print(f"Scanning dataset: {DATASET_ROOT}")

for video_path in Path(DATASET_ROOT).rglob("*"):
    if video_path.suffix.lower() in {".mp4", ".avi", ".mov", ".mkv"}:

        group = video_path.parent.name
        video_id = video_path.stem

        # Use video-level identifiers to prevent frame-level data leakage
        subject = video_id

        video_rows.append(
            {
                "video_path": str(video_path),
                "video_name": video_path.name,
                "subject": subject,
                "group": group,
            }
        )


videos_df = pd.DataFrame(video_rows)

videos_df.to_csv(VIDEOS_INFO_PATH, index=False)

print(f"Saved {len(videos_df)} videos to {VIDEOS_INFO_PATH}")
print(videos_df.head())

print(f"\nUnique subjects: {videos_df['subject'].nunique()}")
print(f"Unique groups: {videos_df['group'].nunique()}")

Scanning dataset: data\raw
Saved 349 videos to data\output\videos_info.csv
                                      video_path                video_name  \
0     data\raw\Dash\Female\1-FemaleNoGlasses.avi     1-FemaleNoGlasses.avi   
1    data\raw\Dash\Female\10-FemaleNoGlasses.avi    10-FemaleNoGlasses.avi   
2  data\raw\Dash\Female\11-FemaleGlasses.avi.avi  11-FemaleGlasses.avi.avi   
3  data\raw\Dash\Female\12-FemaleGlasses.avi.avi  12-FemaleGlasses.avi.avi   
4  data\raw\Dash\Female\13-FemaleGlasses.avi.avi  13-FemaleGlasses.avi.avi   

                subject   group  
0     1-FemaleNoGlasses  Female  
1    10-FemaleNoGlasses  Female  
2  11-FemaleGlasses.avi  Female  
3  12-FemaleGlasses.avi  Female  
4  13-FemaleGlasses.avi  Female  

Unique subjects: 349
Unique groups: 4


## Frame Extraction

The video files are converted into individual frames before training.

Frames are sampled at 10 FPS to reduce redundancy while preserving enough temporal information for the yawning detection task. Metadata for each extracted frame is stored to maintain the connection between frames and their original videos.

In [5]:
from pathlib import Path


FRAMES_OUTPUT_DIR = FRAMES_ROOT / "frames"

FRAMES_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

frames_info = []
TARGET_FPS = 10


for _, row in tqdm(videos_df.iterrows(), total=len(videos_df)):

    video_path = row["video_path"]
    subject = row["subject"]
    video_name = row["video_name"]

    video_id = Path(video_name).stem
    save_dir = FRAMES_OUTPUT_DIR / f"{subject}_{video_id}"

    save_dir.mkdir(parents=True, exist_ok=True)


    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"Could not open video: {video_path}")
        continue


    original_fps = cap.get(cv2.CAP_PROP_FPS) or TARGET_FPS
    frame_step = max(1, round(original_fps / TARGET_FPS))

    frame_index = 0
    saved_frames = 0


    while True:
        ret, frame = cap.read()

        if not ret:
            break


        if frame_index % frame_step == 0:

            frame_path = save_dir / f"frame_{saved_frames:05d}.jpg"

            cv2.imwrite(str(frame_path), frame)

            frames_info.append(
                {
                    "frame_path": str(frame_path),
                    "subject": subject,
                    "video_name": video_name,
                    "frame_index": saved_frames,
                }
            )

            saved_frames += 1


        frame_index += 1


    cap.release()


frames_df = pd.DataFrame(frames_info)

frames_df.to_csv(FRAMES_INFO_PATH, index=False)

print(f"Saved {len(frames_df)} frames to {FRAMES_INFO_PATH}")

frames_df.head()

100%|██████████| 349/349 [07:12<00:00,  1.24s/it]


Saved 96173 frames to data\output\frames_info.csv


,frame_path,subject,video_name,frame_index
0,data\output\frames\frames\1-FemaleNoGlasses_1-...,1-FemaleNoGlasses,1-FemaleNoGlasses.avi,0
1,data\output\frames\frames\1-FemaleNoGlasses_1-...,1-FemaleNoGlasses,1-FemaleNoGlasses.avi,1
2,data\output\frames\frames\1-FemaleNoGlasses_1-...,1-FemaleNoGlasses,1-FemaleNoGlasses.avi,2
3,data\output\frames\frames\1-FemaleNoGlasses_1-...,1-FemaleNoGlasses,1-FemaleNoGlasses.avi,3
4,data\output\frames\frames\1-FemaleNoGlasses_1-...,1-FemaleNoGlasses,1-FemaleNoGlasses.avi,4


## Automatic Frame Labeling

Frame-level labels are generated using facial landmarks extracted with MediaPipe FaceMesh.

The Mouth Aspect Ratio (MAR) is used as a simple indicator of mouth opening. Frames with sustained high MAR values are labeled as yawning frames, while the remaining frames are treated as non-yawning.

Temporal smoothing is applied to reduce noise from individual frame detections.

In [6]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import mediapipe as mp
from tqdm import tqdm


FRAMES_DIR = FRAMES_ROOT / "frames"


mp_face_mesh = mp.solutions.face_mesh


# MediaPipe mouth landmarks
LIP_LEFT = 61
LIP_RIGHT = 291
LIP_TOP = 13
LIP_BOTTOM = 14


def calculate_mar(landmarks, width, height):
    """
    Calculate Mouth Aspect Ratio (MAR) from facial landmarks.
    """

    def point(index):
        return np.array(
            [
                landmarks[index].x * width,
                landmarks[index].y * height,
            ],
            dtype=np.float32,
        )

    left = point(LIP_LEFT)
    right = point(LIP_RIGHT)
    top = point(LIP_TOP)
    bottom = point(LIP_BOTTOM)

    mouth_width = np.linalg.norm(right - left) + 1e-6
    mouth_height = np.linalg.norm(bottom - top)

    return float(mouth_height / mouth_width)

In [7]:
def calculate_mar(landmarks, width, height):
    """
    Calculate Mouth Aspect Ratio (MAR) using facial landmarks.
    """

    def point(index):
        return np.array(
            [
                landmarks[index].x * width,
                landmarks[index].y * height,
            ],
            dtype=np.float32
        )

    left = point(LIP_LEFT)
    right = point(LIP_RIGHT)
    top = point(LIP_TOP)
    bottom = point(LIP_BOTTOM)

    mouth_width = np.linalg.norm(right - left) + 1e-6
    mouth_height = np.linalg.norm(bottom - top)

    return float(mouth_height / mouth_width)

In [8]:
def smooth_signal(values, kernel_size=7):
    values = values.copy()

    if np.all(np.isnan(values)):
        return values

    values[np.isnan(values)] = np.nanmean(values)

    kernel = np.ones(kernel_size) / kernel_size

    return np.convolve(values, kernel, mode="same")

In [9]:
def detect_yawning_frames(
    mar_values,
    threshold=0.35,
    min_duration=8
):
    """
    Detect yawning events based on sustained MAR values.
    """

    above_threshold = mar_values > threshold
    labels = np.zeros_like(above_threshold, dtype=bool)

    start = None

    for index, value in enumerate(above_threshold):

        if value and start is None:
            start = index

        if (not value or index == len(above_threshold)-1) and start is not None:

            end = index if not value else index + 1

            if end - start >= min_duration:
                labels[start:end] = True

            start = None
    return labels

In [10]:
def compute_frame_mar(image, face_mesh):

    height, width = image.shape[:2]

    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    result = face_mesh.process(rgb)

    if not result.multi_face_landmarks:
        return np.nan

    landmarks = result.multi_face_landmarks[0].landmark

    return calculate_mar(landmarks, width, height)

In [11]:
all_rows = []

video_folders = sorted(
    [p for p in FRAMES_DIR.iterdir() if p.is_dir()]
)

with mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
) as face_mesh:

    for video_folder in tqdm(video_folders, desc="Labeling frames"):

        frames = sorted(
            [
                p for p in video_folder.iterdir()
                if p.suffix.lower() in {".jpg", ".png"}
            ]
        )

        mar_values = []
        frame_paths = []

        for frame_path in frames:

            image = cv2.imread(str(frame_path))

            if image is None:
                mar_values.append(np.nan)
            else:
                mar_values.append(
                    compute_frame_mar(image, face_mesh)
                )

            frame_paths.append(frame_path)


        mar_values = np.array(mar_values, dtype=np.float32)

        smoothed_mar = smooth_signal(
            mar_values,
            kernel_size=7
        )

        smoothed_mar = np.nan_to_num(
            smoothed_mar,
            nan=0.0
        )


        yawn_mask = detect_yawning_frames(
            smoothed_mar,
            threshold=0.35,
            min_duration=8
        )


        for path, mar_value, is_yawn in zip(
            frame_paths,
            smoothed_mar,
            yawn_mask
        ):

            all_rows.append(
                {
                    "video_id": video_folder.name,
                    "frame_path": str(path),
                    "mar": float(mar_value),
                    "label": int(is_yawn),
                }
            )


labels_df = pd.DataFrame(all_rows)

LABELS_CSV = OUTPUT_ROOT / "frames_labels.csv"

labels_df.to_csv(
    LABELS_CSV,
    index=False
)

print(f"Saved frame labels to: {LABELS_CSV}")

print("\nLabel distribution:")
print(labels_df["label"].value_counts())

Labeling frames: 100%|██████████| 349/349 [18:54<00:00,  3.25s/it]


Saved frame labels to: data\output\frames_labels.csv

Label distribution:
label
0    89139
1     6304
Name: count, dtype: int64


## Dataset Construction

The detected facial regions are used to create the final training dataset.

Instead of training on complete frames, mouth regions are extracted to focus the model on the visual features related to yawning behavior. The generated crops are stored together with their labels and metadata for later training.

In [12]:
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from pathlib import Path
from tqdm import tqdm


# Paths from previous preprocessing steps
FRAMES_INFO_PATH = FRAMES_INFO_PATH
CROPS_ROOT = CROPS_ROOT
CROPS_INFO_PATH = CROPS_INFO_PATH

CROPS_ROOT.mkdir(parents=True, exist_ok=True)


frames_df = pd.read_csv(FRAMES_INFO_PATH)

LABELS_CSV = OUTPUT_ROOT / "frames_labels.csv"
labels_df = pd.read_csv(LABELS_CSV)

## Merge Frame Labels

The frame metadata and generated labels are combined to create a single dataframe containing all information required for crop extraction.

In [13]:
def normalize_path(path):
    path = str(path).replace("\\", "/")

    if not path.startswith("/"):
        path = str(Path(path).resolve())

    return path


frames_df["frame_path"] = frames_df["frame_path"].apply(normalize_path)
labels_df["frame_path"] = labels_df["frame_path"].apply(normalize_path)


frames_df = frames_df.merge(
    labels_df[["frame_path", "label"]],
    on="frame_path",
    how="left"
)


missing_labels = frames_df["label"].isna().sum()

print(f"Frames loaded: {len(frames_df)}")
print(f"Missing labels: {missing_labels}")

# Remove frames without generated labels
frames_df = frames_df.dropna(subset=["label"])

frames_df["label"] = frames_df["label"].astype(int)

print(f"Frames remaining after removing missing labels: {len(frames_df)}")

print("\nLabel distribution:")
print(frames_df["label"].value_counts())

print("\nLabel distribution:")
print(frames_df["label"].value_counts())

Frames loaded: 96173
Missing labels: 730
Frames remaining after removing missing labels: 95443

Label distribution:
label
0    89139
1     6304
Name: count, dtype: int64

Label distribution:
label
0    89139
1     6304
Name: count, dtype: int64


In [15]:
print(labels_df.columns)
print(labels_df.head())

Index(['video_id', 'frame_path', 'mar', 'label'], dtype='str')
                                            video_id  \
0  1-FemaleNoGlasses-Normal_1-FemaleNoGlasses-Normal   
1  1-FemaleNoGlasses-Normal_1-FemaleNoGlasses-Normal   
2  1-FemaleNoGlasses-Normal_1-FemaleNoGlasses-Normal   
3  1-FemaleNoGlasses-Normal_1-FemaleNoGlasses-Normal   
4  1-FemaleNoGlasses-Normal_1-FemaleNoGlasses-Normal   

                                          frame_path       mar  label  
0  C:\Users\laith\Desktop\UniAssignments\CV\YawDD...  0.002300      0  
1  C:\Users\laith\Desktop\UniAssignments\CV\YawDD...  0.002794      0  
2  C:\Users\laith\Desktop\UniAssignments\CV\YawDD...  0.003397      0  
3  C:\Users\laith\Desktop\UniAssignments\CV\YawDD...  0.003741      0  
4  C:\Users\laith\Desktop\UniAssignments\CV\YawDD...  0.003623      0  


## Facial Landmark Extraction

MediaPipe FaceMesh is used to locate facial landmarks and extract the mouth region from each frame.

In [16]:
mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5
)


LIP_LEFT = 61
LIP_RIGHT = 291
LIP_TOP = 13
LIP_BOTTOM = 14

In [17]:
def extract_mouth_crop(image, landmarks):
    """
    Extract the mouth region using MediaPipe facial landmarks.
    """

    height, width = image.shape[:2]


    def point(index):
        return (
            int(landmarks[index].x * width),
            int(landmarks[index].y * height)
        )


    left_x, left_y = point(LIP_LEFT)
    right_x, right_y = point(LIP_RIGHT)

    top_x, top_y = point(LIP_TOP)
    bottom_x, bottom_y = point(LIP_BOTTOM)


    x_min = max(0, min(left_x, right_x) - 10)
    x_max = min(width, max(left_x, right_x) + 10)

    y_min = max(0, min(top_y, bottom_y) - 10)
    y_max = min(height, max(top_y, bottom_y) + 10)


    if x_max <= x_min or y_max <= y_min:
        return None


    return image[y_min:y_max, x_min:x_max]

## Generate Mouth Crops

Each labeled frame is processed to extract the mouth region. The resulting crops are saved and linked with their original metadata and labels.

In [18]:
crop_rows = []


for _, row in tqdm(
    frames_df.iterrows(),
    total=len(frames_df),
    desc="Extracting mouth crops"
):

    frame_path = row["frame_path"]

    image = cv2.imread(frame_path)

    if image is None:
        continue


    rgb = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    result = face_mesh.process(rgb)


    if not result.multi_face_landmarks:
        continue


    landmarks = result.multi_face_landmarks[0].landmark

    mouth_crop = extract_mouth_crop(
        image,
        landmarks
    )


    if mouth_crop is None or mouth_crop.size == 0:
        continue


    subject_dir = CROPS_ROOT / row["subject"]

    subject_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    crop_name = (
        f"{Path(row['video_name']).stem}_"
        f"{int(row['frame_index']):05d}.jpg"
    )


    crop_path = subject_dir / crop_name


    cv2.imwrite(
        str(crop_path),
        mouth_crop
    )


    crop_rows.append(
        {
            "crop_path": str(crop_path),
            "subject": row["subject"],
            "video_name": row["video_name"],
            "frame_index": row["frame_index"],
            "label": int(row["label"])
        }
    )

Extracting mouth crops: 100%|██████████| 95443/95443 [16:42<00:00, 95.17it/s] 


In [19]:
crops_df = pd.DataFrame(crop_rows)

crops_df.to_csv(
    CROPS_INFO_PATH,
    index=False
)


print(f"Total crops saved: {len(crops_df)}")
print(f"Saved metadata to: {CROPS_INFO_PATH}")

crops_df.head()

Total crops saved: 95238
Saved metadata to: data\output\crops_labeled.csv


,crop_path,subject,video_name,frame_index,label
0,data\output\crops\1-FemaleNoGlasses\1-FemaleNo...,1-FemaleNoGlasses,1-FemaleNoGlasses.avi,0,0
1,data\output\crops\1-FemaleNoGlasses\1-FemaleNo...,1-FemaleNoGlasses,1-FemaleNoGlasses.avi,1,0
2,data\output\crops\1-FemaleNoGlasses\1-FemaleNo...,1-FemaleNoGlasses,1-FemaleNoGlasses.avi,2,0
3,data\output\crops\1-FemaleNoGlasses\1-FemaleNo...,1-FemaleNoGlasses,1-FemaleNoGlasses.avi,3,0
4,data\output\crops\1-FemaleNoGlasses\1-FemaleNo...,1-FemaleNoGlasses,1-FemaleNoGlasses.avi,4,0


In [24]:
print(f"Total crops: {len(crops_df)}")
print("\nLabel distribution:")
print(crops_df["label"].value_counts())

crops_df = pd.read_csv(CROPS_INFO_PATH)

duplicate_labels = (
    crops_df
    .groupby(["subject", "video_name", "frame_index"])["label"]
    .nunique()
)

conflicting_frames = (duplicate_labels > 1).sum()

print(f"Frames with conflicting labels: {conflicting_frames}")

Total crops: 95238

Label distribution:
label
0    88937
1     6301
Name: count, dtype: int64
Frames with conflicting labels: 0


## Dataset Split

The dataset is split at the subject level to prevent data leakage between training, validation, and test sets.

Since multiple frames are extracted from the same video, randomly splitting individual images could result in nearly identical samples appearing in different splits. Splitting by subject ensures a more realistic evaluation.

In [25]:
import numpy as np
import pandas as pd
from pathlib import Path


crops_df = pd.read_csv(CROPS_INFO_PATH)


subjects = crops_df["subject"].unique()

rng = np.random.default_rng(42)
rng.shuffle(subjects)


num_subjects = len(subjects)

train_size = int(0.7 * num_subjects)
val_size = int(0.15 * num_subjects)


train_subjects = subjects[:train_size]

val_subjects = subjects[
    train_size:train_size + val_size
]

test_subjects = subjects[
    train_size + val_size:
]


train_df = crops_df[
    crops_df["subject"].isin(train_subjects)
].reset_index(drop=True)


val_df = crops_df[
    crops_df["subject"].isin(val_subjects)
].reset_index(drop=True)


test_df = crops_df[
    crops_df["subject"].isin(test_subjects)
].reset_index(drop=True)


SPLIT_DIR = OUTPUT_ROOT / "splits"

SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


train_df.to_csv(
    SPLIT_DIR / "train.csv",
    index=False
)

val_df.to_csv(
    SPLIT_DIR / "val.csv",
    index=False
)

test_df.to_csv(
    SPLIT_DIR / "test.csv",
    index=False
)


print(f"Total subjects: {num_subjects}")

print(
    f"Train / Val / Test subjects: "
    f"{len(train_subjects)} / "
    f"{len(val_subjects)} / "
    f"{len(test_subjects)}"
)


print(
    f"Samples: "
    f"train={len(train_df)}, "
    f"val={len(val_df)}, "
    f"test={len(test_df)}"
)

C:\Users\laith\AppData\Local\Temp\ipykernel_32524\587370923.py:12: UserWarning: you are shuffling a 'ArrowStringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(subjects)


Total subjects: 348
Train / Val / Test subjects: 243 / 52 / 53
Samples: train=64367, val=15635, test=15236


In [26]:
assert len(set(train_subjects) & set(val_subjects)) == 0
assert len(set(train_subjects) & set(test_subjects)) == 0
assert len(set(val_subjects) & set(test_subjects)) == 0

print("No subject overlap between splits.")

No subject overlap between splits.


## Dataset Pipeline

A PyTorch dataset is created to load the extracted mouth crops and corresponding labels.

Images are resized, converted into tensors, and normalized before being passed to the model. The dataset is split into training, validation, and test loaders using the subject-level splits created previously.